In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mly.waveforms import WNB
from mly.datatools import DataPod, DataSet, generator

In [ ]:
np.random.seed(111)

### Generating WNB Waveforms

In [ ]:
sample_freq=1024
n=4000
fmin, fmax= 30,400
dmin, dmax= 0.05, 0.5
bandmin, bandmax = 10, 100

pods = []

for i in range(n):
    f_start = fmin+np.random.rand()*(fmax-bandmin-fmin)
    f_end = f_start + np.random.rand()* min(bandmax, fmax-f_start)
    dur = dmin + np.random.rand()*(dmax-dmin)

    #print(f_start, f_end, dur)
    wnb_wf = WNB(duration=dur,
                  fs=sample_freq,
                  fmin=f_start,
                  fmax=f_end,
                  enveloped=True,
                  sidePad=None,
                  hrss= 1)
    wnb_wf=np.asarray(wnb_wf[0]).squeeze()
    pod=DataPod(strain=wnb_wf,
                fs=sample_freq,
                detectors=['I'],
                labels ={ 'type': 'signal',
                         'wf': 'wnb',
                         'fmin': f_start,
                         'fmax': f_end,
                         'dur': dur})
    pods.append(pod)
print(f"Generated {len(pods)} WNB pods")

    


In [ ]:
wnb_dataset = DataSet(pods, name='WNB dataset (single-detector)')
print(wnb_dataset)

In [ ]:
#print(wnb_dataset[0].labels)
#for i in range(15):
 #   wnb_dataset[i].plot()

In [ ]:
wnb_dataset.save('wnb_4000.pkl')

### Injecting WNB Waveforms

In [ ]:
wnb_injections=DataSet.load('wnb_4000.pkl')
wnb_noise_pods = []



for i in range(4000):
    snr=np.random.uniform(5, 50)
    ds = generator(
        duration=1,
        size=1,
        fs=1024,
        detectors=['I'],
        labels={'type': 'signal', 'wf': 'wnb', 'snr': snr},
        backgroundType='optimal',
        windowSize=16,
        injection_source=wnb_injections[i],
        injectionSNR=snr,
        plugins=['psd']  
    )

    wnb_noise_pods.append(ds[0])
wnb_noise_dataset = DataSet(wnb_noise_pods, name='WNB dataset with noise')
print(wnb_noise_dataset)


wnb_noise_dataset.save('wnb_4000_noise.pkl')

In [ ]:
for i in range(5):
    wnb_noise_dataset[i].plot()
    wnb_noise_dataset[i].plot('psd')

### Generating Glitches Using Gengli

In [ ]:
import gengli

g = gengli.glitch_generator('H1')

n_glitch=4000
alpha = 0.2
fhigh=256
fs=1024
def generate_glitches(n, fs, alpha=alpha, fhigh=fhigh):
    #snr=np.random.uniform(5, 50)
    return g.get_glitch(n, srate=fs, alpha=alpha, fhigh=fhigh)

raw_glitches=generate_glitches(n_glitch, fs)
glitch_pods = []
for i in range(n_glitch):
    x=np.asarray(raw_glitches[i]).squeeze()
    pod=DataPod(strain=x,
                fs=fs,
                detectors=['I'],
                labels={'type': 'signal', 'wf': 'blip', 'source': 'gengli'})
    glitch_pods.append(pod)

glitch_dataset = DataSet(glitch_pods, name='Gengli Blip Glitches (80)')
print(glitch_dataset)
print(glitch_dataset[0].labels)

glitch_dataset.save('gengli_blips_4000.pkl') 

for i in range(5):
    glitch_dataset[i].plot()

### Injecting Glitches in Noise

In [ ]:
glitch_injections = DataSet.load('gengli_blips_4000.pkl')
glitch_noise_wnb_pods = []

for i in range(n_glitch):
    snr=np.random.uniform(5, 50)
    ds = generator(
        duration=1,
        size=1,
        fs=1024,
        detectors=['I'],
        labels={'type': 'signal', 'wf': 'blip', 'snr': snr},
        backgroundType='optimal',
        windowSize=16,
        injection_source=glitch_injections[i],
        injectionSNR=snr,
        plugins=['psd']  
    )

    glitch_noise_wnb_pods.append(ds[0])

glitch_noise_dataset = DataSet(glitch_noise_wnb_pods, name='Gengli Blip Glitches with Noise')
print(glitch_noise_dataset)

glitch_noise_dataset.save('gengli_blips_4000_noise.pkl')
for i in range(5):
    glitch_noise_dataset[i].plot()
    glitch_noise_dataset[i].plot('psd')

In [ ]:
glitch_noise_dataset[0].plot('psd')
wnb_noise_dataset[0].plot('psd')

In [ ]:
wnb_ds   = DataSet.load("wnb_4000_noise.pkl")
blip_ds  = DataSet.load("gengli_blips_4000_noise.pkl")

print("WNB:", len(wnb_ds))
print("BLIP:", len(blip_ds))

### Log fft and cnn

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score

def standardize_ts(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    return (x - x.mean()) / (x.std() + 1e-8)

def to_logfft_from_strain(x):
    x = standardize_ts(x)
    fft = np.fft.rfft(x)
    return np.log1p(np.abs(fft)).astype(np.float32)

X_wnb_fft  = np.stack([to_logfft_from_strain(p.strain) for p in wnb_ds])
X_blip_fft= np.stack([to_logfft_from_strain(p.strain) for p in blip_ds])

y_wnb_fft  = np.zeros(len(X_wnb_fft), dtype=np.int64)
y_blip_fft = np.ones(len(X_blip_fft), dtype=np.int64)

X_fft = np.concatenate([X_wnb_fft, X_blip_fft], axis=0)
y_fft = np.concatenate([y_wnb_fft, y_blip_fft], axis=0)

X_fft = X_fft[..., None].astype(np.float32)

print("Final X:", X_fft.shape)

In [ ]:
def build_model_fft(input_len):
    model_fft = models.Sequential([
        layers.Input(shape=(input_len, 1)),
        layers.Conv1D(16, 7, padding="same", activation="relu"),
        layers.MaxPool1D(2),
        layers.GlobalAveragePooling1D(),
        layers.Dense(1, activation="sigmoid"),
    ])
    model_fft.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
    )
    return model_fft

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

TARGET_FPR = 0.05
thr_scores = []
fpr_scores = []
tpr_scores = []   # you should still record this even if you don't obsess over it

for fold, (train_idx, test_idx) in enumerate(skf.split(X_fft, y_fft), 1):
    X_train_full, X_test = X_fft[train_idx], X_fft[test_idx]
    y_train_full, y_test = y_fft[train_idx], y_fft[test_idx]

    # Stratified train/val split (avoid Keras validation_split ordering issues)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2,
        stratify=y_train_full,
        random_state=fold
    )

    model_fft = build_model_fft(input_len=X_train.shape[1])

    model_fft.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=256,
        verbose=0
    )

    # --- choose threshold on TRAIN negatives only ---
    y_prob_train = model_fft.predict(X_train_full, verbose=0).ravel()
    neg_scores = y_prob_train[y_train_full == 0]
    thr = np.quantile(neg_scores, 1 - TARGET_FPR)
    thr_scores.append(thr)

    # --- evaluate on TEST ---
    y_prob_test = model_fft.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob_test >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    fpr_scores.append(fpr)
    tpr_scores.append(tpr)

    print(f"Fold {fold}: thr={thr:.4f} | FPR={fpr:.4f} | TPR={tpr:.4f} | FP={fp}, TN={tn}, TP={tp}, FN={fn}")

print(f"\nMean threshold: {np.mean(thr_scores):.4f} ± {np.std(thr_scores):.4f}")
print(f"Mean FPR:       {np.mean(fpr_scores):.4f} ± {np.std(fpr_scores):.4f}")
print(f"Mean TPR:       {np.mean(tpr_scores):.4f} ± {np.std(tpr_scores):.4f}")

In [ ]:
thr_fft_cv = float(np.mean(thr_scores))
print("average CV thr_fft =", thr_fft_cv)

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(
    X_fft, y_fft,
    test_size=0.2,
    stratify=y_fft,
    random_state=42
)

In [ ]:
trained_model_fft = build_model_fft(input_len=Xtr.shape[1])

trained_model_fft.fit(
    Xtr, ytr,
    validation_data=(Xval, yval),
    epochs=50,
    batch_size=256,
    verbose=0,
    shuffle=True
)

In [ ]:
y_prob_tr = trained_model_fft.predict(Xtr, verbose=0).ravel()
thr_fft = float(np.quantile(y_prob_tr[ytr == 0], 1 - TARGET_FPR))
print("final thr_fft =", thr_fft)

### wnb generation for testing

In [ ]:
sample_freq=1024
n=1000
fmin, fmax= 30,400
dmin, dmax= 0.05, 0.5
bandmin, bandmax = 10, 100

pods = []

for i in range(n):
    f_start = fmin+np.random.rand()*(fmax-bandmin-fmin)
    f_end = f_start + np.random.rand()* min(bandmax, fmax-f_start)
    dur = dmin + np.random.rand()*(dmax-dmin)

    #print(f_start, f_end, dur)
    wnb_wf = WNB(duration=dur,
                  fs=sample_freq,
                  fmin=f_start,
                  fmax=f_end,
                  enveloped=True,
                  sidePad=None,
                  hrss= 1)
    wnb_wf=np.asarray(wnb_wf[0]).squeeze()
    pod=DataPod(strain=wnb_wf,
                fs=sample_freq,
                detectors=['I'],
                labels ={ 'type': 'signal',
                         'wf': 'wnb',
                         'fmin': f_start,
                         'fmax': f_end,
                         'dur': dur})
    pods.append(pod)
print(f"Generated {len(pods)} WNB pods")
wnb_dataset = DataSet(pods, name='WNB dataset (single-detector)')
wnb_dataset.save('wnb_1000.pkl')


### glitch generation for testing

In [ ]:
import gengli

g = gengli.glitch_generator('H1')

n_glitch=1000
alpha = 0.2
fhigh=256
fs=1024
def generate_glitches(n, fs, alpha=alpha, fhigh=fhigh):
    #snr=np.random.uniform(5, 50)
    return g.get_glitch(n, srate=fs, alpha=alpha, fhigh=fhigh)

raw_glitches=generate_glitches(n_glitch, fs)
glitch_pods = []
for i in range(n_glitch):
    x=np.asarray(raw_glitches[i]).squeeze()
    pod=DataPod(strain=x,
                fs=fs,
                detectors=['I'],
                labels={'type': 'signal', 'wf': 'blip', 'source': 'gengli'})
    glitch_pods.append(pod)

glitch_dataset = DataSet(glitch_pods, name='Gengli Blip Glitches (80)')
print(glitch_dataset)
print(glitch_dataset[0].labels)

glitch_dataset.save('gengli_blips_1000.pkl') 

In [ ]:
import numpy as np
import pandas as pd
from mly.datatools import DataSet, generator
from sklearn.metrics import confusion_matrix

SNR_BINS = [(5,10),(10,15),(15,20),(20,25),(25,30),(30,35),(35,40),(40,45),(45,50)]
BIN_MID  = [0.5*(a+b) for a,b in SNR_BINS]

FS = 1024
DUR = 1
DETECTORS = ['I']
WINDOW = 16




wnb_injections   = DataSet.load('wnb_1000.pkl')             
glitch_injections= DataSet.load('gengli_blips_1000.pkl')     
N_PER_CLASS = min(len(wnb_injections), len(glitch_injections))
print("Using N_PER_CLASS =", N_PER_CLASS, "(exhaustive + balanced)")
print(len(wnb_injections))
print( len(glitch_injections))



## evaluation

Assumes you already have: wnb_injections, glitch_injections, trained_model_time, trained_model_fft, thr_time, thr_fft, and your generator params (FS, DUR, DETECTORS, WINDOW).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from mly.datatools import DataSet, generator

# SNR bins 
SNR_BINS = [(5,10),(10,15),(15,20),(20,25),(25,30),(30,35),(35,40),(40,45),(45,50)]
BIN_MID  = [0.5*(a+b) for a,b in SNR_BINS]

# Reproducibility
RNG = np.random.default_rng(519)

N_PER_BIN = 1000   

In [ ]:
def standardize_ts(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    return (x - x.mean()) / (x.std() + 1e-8)

def to_logfft_from_strain(x):
    x = standardize_ts(x)
    fft = np.fft.rfft(x)
    return np.log1p(np.abs(fft)).astype(np.float32)



def dataset_to_X_fft(ds):
    X = np.stack([to_logfft_from_strain(pod.strain) for pod in ds], axis=0)
    return X[..., None].astype(np.float32)  # (N, Lfft, 1)

In [ ]:
def make_noise_injected_dataset_for_bin(bin_low, bin_high, kind="wnb", n_samples=None, rng=None):
    """
    Builds a DataSet of 'noise + injection' examples for a given SNR bin.

    kind='wnb'  -> uses wnb_injections bank (class 0)
    kind='blip' -> uses glitch_injections bank (class 1)

    n_samples=None -> exhaustive (use full bank)
    n_samples=int  -> use exactly that many (no repeats within the bin)
    """
    if rng is None:
        rng = np.random.default_rng()

    if kind == "wnb":
        bank = wnb_injections
        wf_name = "wnb"
    elif kind in ["blip", "glitch"]:
        bank = glitch_injections
        wf_name = "blip"
    else:
        raise ValueError("kind must be 'wnb' or 'blip'/'glitch'")

    n_bank = len(bank)
    if n_samples is None:
        n_use = n_bank
    else:
        n_use = int(min(n_samples, n_bank))

    idx = rng.permutation(n_bank)[:n_use]  

    pods = []
    for i in idx:
        snr = float(rng.uniform(bin_low, bin_high))
        inj = bank[i]

        lab = {
            "type": "signal",
            "wf": wf_name,
            "snr": snr,
            "snr_low": bin_low,
            "snr_high": bin_high,
        }

        ds = generator(
            duration=DUR,
            size=1,
            fs=FS,
            detectors=DETECTORS,
            labels=lab,
            backgroundType="optimal",
            windowSize=WINDOW,
            injection_source=inj,
            injectionSNR=snr,
            plugins=['psd'],  
        )
        pods.append(ds[0])

    name = f"test_{wf_name}_noise_snr_{bin_low}_{bin_high}_n{n_use}"
    return DataSet(pods, name=name)

In [ ]:
def predict_probs(model, X):
    return model.predict(X, verbose=0).ravel()

def confusion_from_scores(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn, fp, fn, tp

def rates_from_confusion(tn, fp, fn, tp):
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return fpr, tpr, acc

In [ ]:
def evaluate_one_bin_fft_only(bin_low, bin_high, n_per_bin=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    ds0 = make_noise_injected_dataset_for_bin(bin_low, bin_high, kind="wnb",  n_samples=n_per_bin, rng=rng)
    ds1 = make_noise_injected_dataset_for_bin(bin_low, bin_high, kind="blip", n_samples=n_per_bin, rng=rng)

    X0_fft = dataset_to_X_fft(ds0)
    X1_fft = dataset_to_X_fft(ds1)

    y_true = np.concatenate([
        np.zeros(len(ds0), dtype=int),
        np.ones(len(ds1), dtype=int)
    ])

    y_prob_fft = np.concatenate([
        trained_model_fft.predict(X0_fft, verbose=0).ravel(),
        trained_model_fft.predict(X1_fft, verbose=0).ravel()
    ])

    y_pred = (y_prob_fft >= thr_fft).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return {
        "snr_low": bin_low, "snr_high": bin_high, "snr_mid": 0.5*(bin_low + bin_high),
        "n0": len(ds0), "n1": len(ds1),
        "thr_fft": float(thr_fft),
        "fpr_fft": float(fpr), "tpr_fft": float(tpr), "acc_fft": float(acc),
        "tn_fft": int(tn), "fp_fft": int(fp), "fn_fft": int(fn), "tp_fft": int(tp),
    }

In [ ]:
results = []
for (a, b) in SNR_BINS:
    print(f"Evaluating bin {a}-{b} ...")
    out = evaluate_one_bin_fft_only(a, b, n_per_bin=N_PER_BIN, rng=RNG)
    print(f"FFT: FPR={out['fpr_fft']:.3f}, TPR={out['tpr_fft']:.3f}")
    results.append(out)

df = pd.DataFrame(results)
display(df)

df.to_csv("binned_eval_results.csv", index=False)
print("Saved: binned_eval_results.csv")

In [ ]:
plt.figure()
plt.plot(df["snr_mid"], df["fpr_fft"],  marker="o", label="FPR (log-FFT)")
plt.xlabel("SNR Bin Midpoints")
plt.ylabel("False Positive Rate (FPR)")
plt.grid(True)
plt.show()


In [ ]:

print("Mean FPR across bins (fft): ", df["fpr_fft"].mean())

In [ ]:
average_tpr_fft = df["tpr_fft"].mean()
print("Mean TPR across bins (fft): ", average_tpr_fft)

In [ ]:
df_table = df.copy()

df_table["snr_bin"] = df_table.apply(lambda r: f"{int(r['snr_low'])}-{int(r['snr_high'])}", axis=1)

df_table = df_table[["snr_bin", "fpr_fft", "acc_fft"]].rename(
    columns={"snr_bin": "SNR bin", "fpr_fft": "FPR", "acc_fft": "Accuracy"}
)


df_table["FPR"] = df_table["FPR"].map(lambda x: f"{x:.3f}")
df_table["Accuracy"] = df_table["Accuracy"].map(lambda x: f"{x:.3f}")

display(df_table)

In [ ]:
df_table.to_csv("table_fft_fpr_accuracy.csv", index=False)
print("Saved: table_fft_fpr_accuracy.csv")

latex = df_table.to_latex(index=False, escape=False, column_format="lcc")
print(latex)

In [ ]:
SNR_EXAMPLE = 30.0
idx_wnb = 0
idx_blip = 0

ds_wnb = generator(
    duration=1, size=1, fs=FS, detectors=DETECTORS,
    labels={'type': 'signal', 'wf': 'wnb', 'snr': SNR_EXAMPLE},
    backgroundType="optimal", windowSize=WINDOW,
    injection_source=wnb_injections[idx_wnb], injectionSNR=SNR_EXAMPLE,
    plugins=['psd']
)

ds_blip = generator(
    duration=1, size=1, fs=FS, detectors=DETECTORS,
    labels={'type': 'signal', 'wf': 'blip', 'snr': SNR_EXAMPLE},
    backgroundType="optimal", windowSize=WINDOW,
    injection_source=glitch_injections[idx_blip], injectionSNR=SNR_EXAMPLE,
    plugins=['psd']
)

x_wnb  = np.asarray(ds_wnb[0].strain).reshape(-1)
x_blip = np.asarray(ds_blip[0].strain).reshape(-1)

t = np.arange(len(x_wnb)) / FS
print("Shapes:", x_wnb.shape, x_blip.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.plot(t, x_wnb)
plt.xlabel("Time (s)")
plt.ylabel("Strain")
plt.title(f"WNB + noise (SNR={SNR_EXAMPLE:g})")
plt.grid(True)
plt.tight_layout()
plt.savefig("example_wnb_time.png", dpi=300)
plt.show()

plt.figure()
plt.plot(t, x_blip)
plt.xlabel("Time (s)")
plt.ylabel("Strain")
plt.title(f"Blip + noise (SNR={SNR_EXAMPLE:g})")
plt.grid(True)
plt.tight_layout()
plt.savefig("example_blip_time.png", dpi=300)
plt.show()

In [ ]:
def logfft(x):
    x = np.asarray(x, dtype=np.float32)
    x = (x - x.mean()) / (x.std() + 1e-8)
    X = np.fft.rfft(x)
    return np.log1p(np.abs(X))

f = np.fft.rfftfreq(len(x_wnb), d=1/FS)

W_wnb  = logfft(x_wnb)
W_blip = logfft(x_blip)

plt.figure()
plt.plot(f, W_wnb)
plt.xlim(0, FS/2)
plt.xlabel("Frequency (Hz)")
plt.ylabel("log(1 + |FFT|)")
plt.title(f"WNB + noise FFT (SNR={SNR_EXAMPLE:g})")
plt.grid(True)
plt.tight_layout()
plt.savefig("example_wnb_fft.png", dpi=300)
plt.show()

plt.figure()
plt.plot(f, W_blip)
plt.xlim(0, FS/2)
plt.xlabel("Frequency (Hz)")
plt.ylabel("log(1 + |FFT|)")
plt.title(f"Blip + noise FFT (SNR={SNR_EXAMPLE:g})")
plt.grid(True)
plt.tight_layout()
plt.savefig("example_blip_fft.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = df["snr_mid"].to_numpy()
p = df["fpr_fft"].to_numpy()
n_neg = df["n0"].to_numpy()  # should be 1000 everywhere

TARGET_FPR = 0.05
se = np.sqrt(np.clip(p * (1 - p) / n_neg, 0, None))

y_min = 0.0
y_max = max(0.10, float(np.max(p + se) + 0.01))
y_max = min(y_max, 0.30)

plt.figure(figsize=(6.2, 4.2))
plt.errorbar(x, p, yerr=se, fmt="o-", linewidth=1.6, capsize=3, label="Measured FPR (log-FFT)")
plt.axhline(TARGET_FPR, linestyle="--", linewidth=1.4, label="Target FPR = 0.05")

plt.xlabel("SNR bin midpoint")
plt.ylabel("False Positive Rate (FPR)")
plt.ylim(y_min, y_max)
plt.xlim(min(x) - 1, max(x) + 1)

plt.grid(True, alpha=0.35)
plt.legend(frameon=True)
plt.tight_layout()

plt.savefig("fpr_vs_snr_logfft_errorbars.png", dpi=400)
plt.savefig("fpr_vs_snr_logfft_errorbars.pdf")
plt.show()